In [1]:
from pulao.bar.sbar import SBar
from pulao.bar.cbar_manager import CBarManager
from pulao.bar.sbar_manager import SBarManager
from pulao.swing.swing_manager import SwingManager
from pulao.trend.trend_manager import TrendManager
import logging

from pulao.keyzone.keyzone_manager import KeyZoneManager
from pulao.logging import init_logging
from pulao.mtc import MultiTimeframeContext

from pulao.constant import Timeframe

import polars as pl

from pulao.logging import logger

init_logging(level=logging.DEBUG)
open("./logs/pulao.log", "w").close()  # 每次运行前清空日志
logger.debug("开始运行")

df = pl.read_csv("../dataset/I8888.XDCE_60m.csv", try_parse_dates=True)
df = df.slice(0,100)  # test

sbar_list = []
columns = df.columns

for idx, row in enumerate(df.iter_rows()):
    row_dict = dict(zip(columns, row))
    # datetime,open,close,high,low,volume,money,open_interest,signal
    datetime = row_dict["datetime"]
    o = row_dict["open"]
    close = row_dict["close"]
    high = row_dict["high"]
    low = row_dict["low"]
    volume = row_dict["volume"]
    money = row_dict["money"]
    open_interest = row_dict["open_interest"]

    sbar = SBar(
        symbol="i8888",
        exchange="shfe",
        timeframe=Timeframe.H1,
        datetime=datetime,
        turnover=money,
        open_price=o,
        close_price=close,
        high_price=high,
        low_price=low,
        volume=volume,
    )

    sbar_list.append(sbar)
# 模拟行情数据接收
sbar_manager = SBarManager()
cbar_manager = CBarManager(sbar_manager=sbar_manager)
swing_manager = SwingManager(cbar_manager=cbar_manager)
trend_manager = TrendManager(swing_manager=swing_manager)
# 周期：manager list
mtc = MultiTimeframeContext()
mtc.register(Timeframe.M15, sbar_manager, cbar_manager, swing_manager, trend_manager)
mtc.register(Timeframe.H1, sbar_manager, cbar_manager, swing_manager, trend_manager)
mtc.register(Timeframe.D1, sbar_manager, cbar_manager, swing_manager, trend_manager)
keyzone = KeyZoneManager(mtc=mtc)

# 注入模拟数据
for sbar in sbar_list:
    sbar_manager.append(sbar)

logger.debug("运行结束")


{"event": "开始运行", "level": "debug", "logger": "__main__", "time": "2025-12-05 02:10:46"}
{"cbar_count": 1, "event": "用于组成分形的cbar数量不够", "level": "info", "logger": "__main__", "time": "2025-12-05 02:10:46"}
{"swing_list": null, "event": "波段数量不足3个，跳过", "level": "debug", "logger": "__main__", "time": "2025-12-05 02:10:46"}
{"event": "_on_mtc: 1h, trend.changed, {'backtrack_id': None}", "level": "debug", "logger": "__main__", "time": "2025-12-05 02:10:46"}
{"event": "_on_mtc: 1h, swing.changed, {'backtrack_id': None}", "level": "debug", "logger": "__main__", "time": "2025-12-05 02:10:46"}
{"event": "_on_mtc: 1h, cbar.changed, {'backtrack_id': None}", "level": "debug", "logger": "__main__", "time": "2025-12-05 02:10:46"}
{"event": "_on_mtc: 1h, bar.created, {'sbar': SBar(id=122520007157481472, symbol='i8888', exchange='shfe', timeframe=1h, datetime=datetime.datetime(2024, 1, 22, 14, 0), volume=16884.0, turnover=1582169600.0, open_interest=0, open_price=937.425, high_price=939.407, low_price=

In [2]:
from typing import Optional
from pulao.constant import *

mtc.tf_mgr

{15m: {'sbar_manager': <pulao.bar.sbar_manager.SBarManager at 0x2085709dbe0>,
  'cbar_manager': <pulao.bar.cbar_manager.CBarManager at 0x2085709e3c0>,
  'swing_manager': <pulao.swing.swing_manager.SwingManager at 0x2085709e510>,
  'trend_manager': <pulao.trend.trend_manager.TrendManager at 0x2085709e660>},
 1h: {'sbar_manager': <pulao.bar.sbar_manager.SBarManager at 0x2085709dbe0>,
  'cbar_manager': <pulao.bar.cbar_manager.CBarManager at 0x2085709e3c0>,
  'swing_manager': <pulao.swing.swing_manager.SwingManager at 0x2085709e510>,
  'trend_manager': <pulao.trend.trend_manager.TrendManager at 0x2085709e660>},
 1d: {'sbar_manager': <pulao.bar.sbar_manager.SBarManager at 0x2085709dbe0>,
  'cbar_manager': <pulao.bar.cbar_manager.CBarManager at 0x2085709e3c0>,
  'swing_manager': <pulao.swing.swing_manager.SwingManager at 0x2085709e510>,
  'trend_manager': <pulao.trend.trend_manager.TrendManager at 0x2085709e660>}}

In [2]:
# 画图
import polars as pl
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pulao.constant import *

# sbar_manager = SBarManager()
# cbar_manager = CBarManager(sbar_manager=sbar_manager)
# swing_manager = SwingManager(cbar_manager=cbar_manager)
# trend_manager = TrendManager(swing_manager=cbar_manager)
# cbar_manager.read_parquet()
# swing_manager.read_parquet()
# trend_manager.read_parquet()
#
# region . Plotly - Cbar
#
df_cbar = cbar_manager.df_cbar.to_pandas()
df_cbar["datetime"] = pd.date_range("2025-01-01", periods=df_cbar.shape[0], freq="h")
df_cbar["open_price"] = df_cbar["high_price"]
df_cbar["close_price"] = df_cbar["low_price"]
if "index" in df_cbar.columns:
    del df_cbar["index"]

df_cbar.insert(0, "index", range(len(df_cbar)))

# region 构建波段数据
df_swing = swing_manager.df_swing.to_pandas()
df_swing["start_datetime"] = pd.Series(dtype="datetime64[ns]")
df_swing["end_datetime"] = pd.Series(dtype="datetime64[ns]")
if "index" in df_swing.columns:
    del df_swing["index"]
df_swing.insert(0, "index", range(len(df_swing)))

for i, row in enumerate(df_swing.itertuples()):
    df = df_cbar[
        (df_cbar["id"] >= row.cbar_start_id) & (df_cbar["id"] <= row.cbar_end_id)
    ]
    start_datetime = df.iloc[0]["datetime"]
    end_datetime = df.iloc[-1]["datetime"]
    df_swing.at[i, "start_datetime"] = start_datetime
    df_swing.at[i, "end_datetime"] = end_datetime

xs_swing = []
ys_swing = []
texts_swing = []
text_positions_swing = []
for i, row in enumerate(df_swing.itertuples()):
    xs_swing += [row.start_datetime, row.end_datetime, None]
    print("df_swing - "+str(i),row)
    start_index = df_cbar[(df_cbar["id"] == row.cbar_start_id)]["index"].iloc[0]
    end_index = df_cbar[(df_cbar["id"] == row.cbar_end_id)]["index"].iloc[0]
    if row.direction == Direction.UP:
        start_price = df_cbar[(df_cbar["id"] == row.cbar_start_id)]["low_price"].iloc[0]
        end_price = df_cbar[(df_cbar["id"] == row.cbar_end_id)]["high_price"].iloc[0]
        text_positions_swing += ["bottom center", "top center", "middle center"]
    else:
        start_price = df_cbar[(df_cbar["id"] == row.cbar_start_id)]["high_price"].iloc[
            0
        ]
        end_price = df_cbar[(df_cbar["id"] == row.cbar_end_id)]["low_price"].iloc[0]
        text_positions_swing += ["top center", "bottom center", "middle center"]
    ys_swing += [start_price, end_price, None]
    texts_swing += [start_index, end_index, None]
# endregion

# region 构建趋势数据
df_trend = trend_manager.df_trend.to_pandas()
df_trend["start_datetime"] = pd.Series(dtype="datetime64[ns]")
df_trend["end_datetime"] = pd.Series(dtype="datetime64[ns]")
df_trend.insert(0, "index", range(len(df_trend)))

for i, row in enumerate(df_trend.itertuples()):
    print("df_trend[datatime] - "+ str(i), row)
    df = df_swing[
        (df_swing["id"] >= row.swing_start_id) & (df_swing["id"] <= row.swing_end_id)
    ]
    start_datetime = df.iloc[0]["start_datetime"]
    end_datetime = df.iloc[-1]["end_datetime"]
    df_trend.at[i, "start_datetime"] = start_datetime
    df_trend.at[i, "end_datetime"] = end_datetime

xs_trend = []
ys_trend = []
texts_trend = []
text_positions_trend = []
for i, row in enumerate(df_trend.itertuples()):
    xs_trend += [row.start_datetime, row.end_datetime, None]
    print("df_trend[marker] - "+ str(i), row)
    start_index = df_swing[(df_swing["id"] == row.swing_start_id)]["index"].iloc[0]
    end_index = df_swing[(df_swing["id"] == row.swing_end_id)]["index"].iloc[0]
    if row.direction == Direction.UP:
        start_price = df_swing[(df_swing["id"] == row.swing_start_id)]["low_price"].iloc[0]
        end_price = df_swing[(df_swing["id"] == row.swing_end_id)]["high_price"].iloc[0]
        text_positions_trend += ["bottom center", "top center", "middle center"]
    else:
        start_price = df_swing[(df_swing["id"] == row.swing_start_id)]["high_price"].iloc[
            0
        ]
        end_price = df_swing[(df_swing["id"] == row.swing_end_id)]["low_price"].iloc[0]
        text_positions_trend += ["top center", "bottom center", "middle center"]
    ys_trend += [start_price, end_price, None]
    texts_trend += [start_index, end_index, None]
#endregion

# region fig
fig = make_subplots(rows=1, cols=1)
fig.add_trace(
    go.Candlestick(
        x=df_cbar["datetime"],
        open=df_cbar["open_price"],
        high=df_cbar["high_price"],
        low=df_cbar["low_price"],
        close=df_cbar["close_price"],
        name="K线",
    ),
    row=1,
    col=1,
)
#
# df_fractal_bottom = df_cbar[(df_cbar["fractal_type"] == FractalType.BOTTOM)]
#
# fig.add_trace(
#     go.Scatter(
#         x=df_fractal_bottom["datetime"],
#         y=df_fractal_bottom["low_price"] * 0.995,  # 放在K线最低价下方一点，不遮挡蜡烛
#         mode="markers+text",
#         name="swing point low",
#         marker=dict(
#             symbol="triangle-up",
#             size=14,
#             color="#1E90FF",
#         ),
#         text=df_fractal_bottom["index"],  # ← 添加序号
#         textposition="bottom center",  # ← 文字位置
#         textfont=dict(size=10, color="white"),
#         hovertemplate="<b>波段低点</b><br>"
#         + "时间: %{x}<br>"
#         + "价格: %{y:.2f}<br>"
#         + "<extra></extra>",
#     ),
#     row=1,
#     col=1,
# )
#
#
# df_fractal_top = df_cbar[(df_cbar["fractal_type"] == FractalType.TOP)]
#
# fig.add_trace(
#     go.Scatter(
#         x=df_fractal_top["datetime"],
#         y=df_fractal_top["high_price"] * 1.005,  # 放在K线最高价上方一点
#         mode="markers+text",
#         name="swing point high",
#         marker=dict(
#             symbol="triangle-down",
#             size=14,
#             color="#FF4500",
#         ),
#         text=df_fractal_top["index"],  # ← 添加序号
#         textposition="top center",  # ← 文字位置
#         textfont=dict(size=10, color="white"),
#         hovertemplate="<b>波段高点</b><br>"
#         + "时间: %{x}<br>"
#         + "价格: %{y:.2f}<br>"
#         + "<extra></extra>",
#     ),
#     row=1,
#     col=1,
# )

fig.add_trace(
    go.Scatter(
        x=xs_swing,
        y=ys_swing,
        name="swing segments",
        mode="lines",  # lines+text 支持显示文字
        line=dict(width=2, color="orange"),
        text=texts_swing,
        textposition=text_positions_swing,  # 文字位置
    )
)

fig.add_trace(
    go.Scatter(
        x=xs_trend,
        y=ys_trend,
        name="trend segments",
        mode="lines",  # lines+text 支持显示文字
        line=dict(width=2, color="red"),
        text=texts_trend,
        textposition=text_positions_trend,  # 文字位置
    )
)

title = "Swing Chart"
fig.update_layout(
    title=title,
    height=900,
    hovermode="x unified",  # X轴悬停联动虚线
)

fig.update_xaxes(
    showgrid=False,
)

fig.update_yaxes(
    showgrid=False,
)

fig.show()
# endregion

# endregion

df_swing - 0 Pandas(Index=0, index=0, id=122485266626658304, cbar_start_id=122485265880059904, cbar_end_id=122485266597285888, sbar_start_id=122485265875861504, sbar_end_id=122485266555338752, high_price=988.5280151367188, low_price=930.2979736328125, direction=1, is_completed=True, created_at=Timestamp('2025-12-04 23:52:43.230000'), start_datetime=Timestamp('2025-01-01 01:00:00'), end_datetime=Timestamp('2025-01-02 00:00:00'))
df_swing - 1 Pandas(Index=1, index=1, id=122485267859783680, cbar_start_id=122485266597285888, cbar_end_id=122485267813634048, sbar_start_id=122485266555338752, sbar_end_id=122485267788464128, high_price=988.5280151367188, low_price=906.1430053710938, direction=-1, is_completed=True, created_at=Timestamp('2025-12-04 23:52:43.524000'), start_datetime=Timestamp('2025-01-02 00:00:00'), end_datetime=Timestamp('2025-01-03 04:00:00'))
df_swing - 2 Pandas(Index=2, index=2, id=122485268283408384, cbar_start_id=122485267813634048, cbar_end_id=122485268245647360, sbar_sta

In [7]:
trend_manager.df_trend
format_df_trend()

NameError: name 'format_df_trend' is not defined

In [9]:
from typing import List

# 测试构建算法
from IPython.display import display
from pulao.logging import logger, init_logging
import logging
from time import sleep
from pulao.swing import Swing, SwingManager
from pulao.trend import Trend, TrendManager
from pulao.bar import CBar, CBarManager, SBarManager
from pulao.constant import *
import polars as pl

init_logging(level=logging.DEBUG)
with open("./logs/pulao.log", "w"):
    pass  # 每次运行前清空日志

logger.debug("开始运行")


def format_df_trend():
    import polars as pl

    if "index" in swing_manager.df_swing.columns:
        swing_manager.df_swing = swing_manager.df_swing.drop("index")
    swing_manager.df_swing = swing_manager.df_swing.with_row_index("index")

    df_swing_tmp = swing_manager.df_swing.select(["id", "index"])
    df1 = trend_manager.df_trend.join(
        df_swing_tmp, left_on="swing_start_id", right_on="id", how="left"
    )
    df2 = trend_manager.df_trend.join(
        df_swing_tmp, left_on="swing_end_id", right_on="id", how="left"
    )
    df1 = df1.rename({"index": "swing_start_index"})
    df2 = df2.rename({"index": "swing_end_index"})
    df2 = df2.select(["id", "swing_end_index"])
    df1 = df1.join(df2, on="id", how="left")
    df = df1.select(
        [
            "id",
            "swing_start_index",
            "swing_end_index",
            "swing_start_id",
            "swing_end_id",
            "high_price",
            "low_price",
            "direction",
            "is_completed",
            "created_at",
        ]
    )

    return df


cbar_manager = CBarManager(sbar_manager=SBarManager())
swing_manager = SwingManager(cbar_manager=cbar_manager)
trend_manager = TrendManager(swing_manager=swing_manager)
df_cbar = cbar_manager.read_parquet()
df_swing = swing_manager.read_parquet()
df_swing = trend_manager.read_parquet()
# df_swing = df_swing.slice(18,20)
# swing_manager.df_swing = swing_manager.df_swing.slice(0, 0)  # 清空原数据
# trend_manager.df_trend = trend_manager.df_trend.slice(0, 0)
# for i in range(df_swing.height - 1):
#     swing = Swing(**df_swing.row(i, named=True))
#     new_swing = {
#             "id": swing.id,
#             "direction": swing.direction.value,
#             "cbar_start_id": swing.cbar_start_id,
#             "cbar_end_id": swing.cbar_end_id,
#             "sbar_start_id": swing.sbar_start_id,
#             "sbar_end_id": swing.sbar_end_id,
#             "high_price": swing.high_price,
#             "low_price": swing.low_price,
#             "is_completed": swing.is_completed,
#             "created_at": swing.created_at,
#         }
#
#     swing_manager.df_swing = swing_manager.df_swing.vstack(
#         pl.DataFrame([new_swing], schema=swing_manager.df_swing.schema)
#     )
#     swing_manager.notify(EventType.SWING_CHANGED, payload=dict(backtrack_id=None))


logger.debug("运行结束")
sleep(0.2)


display("df_trend")
df = format_df_trend()
display(df)

display("df_swing")
swing_manager.df_swing

{"event": "开始运行", "level": "debug", "logger": "__main__", "time": "2025-12-03 16:34:42"}
{"event": "运行结束", "level": "debug", "logger": "__main__", "time": "2025-12-03 16:34:42"}


'df_trend'

id,swing_start_index,swing_end_index,swing_start_id,swing_end_id,high_price,low_price,direction,is_completed,created_at
u64,u32,u32,u64,u64,f32,f32,i8,bool,datetime[ms]
122008733705314304,1,9,122008716089233408,122008723685117952,988.528015,731.429016,-1,true,2025-12-03 16:19:08.928
122008734015692800,10,16,122008727548071936,122008732300218368,923.666016,731.429016,1,true,2025-12-03 16:19:09.002
122008759303151616,17,39,122008733055193088,122008758753693696,923.666016,657.40802,-1,true,2025-12-03 16:19:15.031
122008779339341824,40,40,122008760431415296,122008760431415296,863.708984,657.40802,1,true,2025-12-03 16:19:19.808
122008783177129984,41,43,122008767540760576,122008773605724160,842.429993,732.270996,-1,true,2025-12-03 16:19:20.723
122008801527209984,44,56,122008778232041472,122008801187467264,835.335022,732.270996,1,true,2025-12-03 16:19:25.098
122008801728536576,56,56,122008801187467264,122008801187467264,835.335022,741.731995,-1,false,2025-12-03 16:19:25.146


'df_swing'

index,id,cbar_start_id,cbar_end_id,sbar_start_id,sbar_end_id,high_price,low_price,direction,is_completed,created_at
u32,u64,u64,u64,u64,u64,f32,f32,i8,bool,datetime[ms]
0,122008713740423168,122008712511488000,122008713694281728,122008712511488000,122008713606201344,988.528015,930.297974,1,true,2025-12-03 16:19:04.168
1,122008716089233408,122008713694281728,122008716026314752,122008713606201344,122008716001148928,988.528015,906.143005,-1,true,2025-12-03 16:19:04.728
2,122008716689018880,122008716026314752,122008716659654656,122008716001148928,122008716659654656,955.924988,906.143005,1,true,2025-12-03 16:19:04.871
3,122008718127665152,122008716659654656,122008718089912320,122008716659654656,122008718089912320,955.924988,842.133972,-1,true,2025-12-03 16:19:05.214
4,122008718316408832,122008718089912320,122008718261878784,122008718089912320,122008718257684480,879.856995,842.133972,1,true,2025-12-03 16:19:05.259
…,…,…,…,…,…,…,…,…,…,…
53,122008790030618624,122008787816022016,122008789938339840,122008787803439104,122008789921562624,786.112,741.731995,-1,true,2025-12-03 16:19:22.357
54,122008797580365824,122008789938339840,122008797479698432,122008789921562624,122008797454532608,824.000977,741.731995,1,true,2025-12-03 16:19:24.157
55,122008799442636800,122008797479698432,122008799232917504,122008797454532608,122008799044173824,824.000977,782.697998,-1,true,2025-12-03 16:19:24.601


In [ ]:
trend_manager.pretty_worker_id()

In [ ]:
import polars as pl
import datetime
swing_manager.df_swing.filter(pl.col("id")>=121676756103987200)
display(trend_manager.active_trend_sfs.sfs)

trend_manager.active_trend_sfs.get_fractal_type()
right = trend_manager.active_trend_sfs.sfs[-1]
mid = trend_manager.active_trend_sfs.sfs[-2]
left = trend_manager.active_trend_sfs.sfs[-3]
if left.low_price> mid.low_price and right.low_price > mid.low_price:
    print("ok")

In [ ]:
# 可视化日志
import pandas as pd

df = pd.read_json("./logs/pulao.log", lines=True)
df = df.drop(columns=["logger", "level", "time"])
priority_cols = ["event"]
other_cols = [c for c in df.columns if c not in priority_cols]
df = df[priority_cols + other_cols]
df["info"] = df[other_cols].apply(lambda row: row.dropna().tolist(), axis=1)
df = df.drop(columns=other_cols)
df["info"] = df["info"].apply(lambda x: "" if x == [] else x)
df